In [31]:
import json
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from safetensors.torch import load_file as load_safetensors_file
except Exception:
    load_safetensors_file = None

BASE_DIR = Path.cwd().resolve()
MODELS_BASE = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'results' / 'model_diagnostics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    ('1_baseline', 'bert_9classes_final'),
    ('2_groupdro', 'bert_gdro_eta01_2ep'),
    ('3_scrubbing', 'bert_scrubbing'),
    ('4_oversampling', 'bert_oversample_only'),
    ('5_focal_loss', 'bert_focal_loss'),
    ('6_adversarial', 'bert_adversarial'),
    ('7_label_smoothing', 'bert_label_smoothing'),
    ('8_attribution_reg', 'bert_attr_reg'),
    ('9_combined_scrub_gdro', 'bert_debiased_combo'),
    ('10_combined_best', 'combined_scrubbing_groupdro'),
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Models base: {MODELS_BASE.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')


Device: cpu
Models base: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models
Output dir: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/model_diagnostics


In [32]:
def find_model_layout(model_dir):
    model_dir = Path(model_dir)
    if not model_dir.exists():
        return None, None
    if (model_dir / 'config.json').exists():
        return 'hf_root', model_dir
    if (model_dir / 'bert' / 'config.json').exists():
        return 'split_bert', model_dir
    candidates = sorted([sub for sub in model_dir.iterdir() if sub.is_dir() and (sub / 'config.json').exists()])
    if candidates:
        preferred = [sub for sub in candidates if sub.name == 'final']
        return 'nested_hf', preferred[0] if preferred else candidates[-1]
    return None, None


def find_weight_file(path):
    path = Path(path)
    for name in ['model.safetensors', 'pytorch_model.bin']:
        candidate = path / name
        if candidate.exists():
            return candidate
    return None


def read_state_keys(weight_file):
    if weight_file is None:
        return []
    weight_file = Path(weight_file)
    if weight_file.suffix == '.safetensors':
        if load_safetensors_file is None:
            raise RuntimeError('safetensors is not available')
        state = load_safetensors_file(str(weight_file), device='cpu')
        return sorted(state.keys())
    obj = torch.load(weight_file, map_location='cpu')
    if isinstance(obj, dict) and 'state_dict' in obj and isinstance(obj['state_dict'], dict):
        return sorted(obj['state_dict'].keys())
    if isinstance(obj, dict):
        return sorted(obj.keys())
    return []


def extract_classifier_keys(keys):
    direct = [k for k in keys if k.startswith('classifier.')]
    bare = [k for k in keys if k in {'weight', 'bias'}]
    return sorted(set(direct + bare))


def sidecar_candidates(model_dir):
    model_dir = Path(model_dir)
    return [model_dir / 'classifier.pt', model_dir / 'model.pt', model_dir / 'classifier.bin']


def try_load(layout, resolved_path):
    tokenizer = None
    model = None
    loading_info = {}
    missing_keys = []
    error = ''
    try:
        if layout == 'split_bert':
            bert_path = resolved_path / 'bert'
            tokenizer_path = resolved_path if (resolved_path / 'tokenizer.json').exists() else bert_path
            tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(bert_path), output_loading_info=True)
        else:
            tokenizer = AutoTokenizer.from_pretrained(str(resolved_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(resolved_path), output_loading_info=True)
        missing_keys = loading_info.get('missing_keys', []) or []
        model = model.to(device).eval()
        return True, tokenizer is not None, missing_keys, error
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
        return False, tokenizer is not None, missing_keys, error
    finally:
        del tokenizer
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_model_checks(model_name, folder_name):
    model_dir = MODELS_BASE / folder_name
    layout, resolved_path = find_model_layout(model_dir)
    result = {
        'model_name': model_name,
        'folder_name': folder_name,
        'model_dir_exists': model_dir.exists(),
        'layout': layout or 'missing',
        'resolved_path': str(resolved_path) if resolved_path else '',
        'config_exists': False,
        'main_weight_file': '',
        'main_classifier_keys': '',
        'sidecar_file': '',
        'sidecar_classifier_keys': '',
        'label_encoder_found': False,
        'tokenizer_loaded': False,
        'load_ok': False,
        'missing_classifier_on_load': False,
        'missing_keys': '',
        'error': '',
    }
    if resolved_path is None:
        return result

    config_path = resolved_path / 'config.json' if layout != 'split_bert' else resolved_path / 'bert' / 'config.json'
    result['config_exists'] = config_path.exists()

    main_weights_base = resolved_path if layout != 'split_bert' else resolved_path / 'bert'
    main_weight_file = find_weight_file(main_weights_base)
    result['main_weight_file'] = str(main_weight_file) if main_weight_file else ''
    if main_weight_file:
        try:
            result['main_classifier_keys'] = ', '.join(extract_classifier_keys(read_state_keys(main_weight_file)))
        except Exception as exc:
            result['main_classifier_keys'] = f'ERROR: {type(exc).__name__}: {exc}'

    sidecar_file = ''
    for candidate in sidecar_candidates(model_dir):
        if candidate.exists():
            sidecar_file = candidate
            break
    result['sidecar_file'] = str(sidecar_file) if sidecar_file else ''
    if sidecar_file:
        try:
            result['sidecar_classifier_keys'] = ', '.join(extract_classifier_keys(read_state_keys(sidecar_file)))
        except Exception as exc:
            result['sidecar_classifier_keys'] = f'ERROR: {type(exc).__name__}: {exc}'

    label_encoder_paths = [resolved_path / 'label_encoder.joblib', resolved_path.parent / 'label_encoder.joblib', model_dir / 'label_encoder.joblib']
    result['label_encoder_found'] = any(path.exists() for path in label_encoder_paths)

    load_ok, tokenizer_loaded, missing_keys, error = try_load(layout, resolved_path)
    result['tokenizer_loaded'] = tokenizer_loaded
    result['load_ok'] = load_ok
    result['missing_keys'] = ', '.join(missing_keys)
    result['missing_classifier_on_load'] = any(key.startswith('classifier.') for key in missing_keys)
    result['error'] = error
    return result


In [33]:
result_1 = run_model_checks('1_baseline', 'bert_9classes_final')
pd.DataFrame([result_1]).to_csv(OUTPUT_DIR / '1_baseline.csv', index=False)
pd.DataFrame([result_1])


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 160.35it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,1_baseline,bert_9classes_final,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [34]:
result_2 = run_model_checks('2_groupdro', 'bert_gdro_eta01_2ep')
pd.DataFrame([result_2]).to_csv(OUTPUT_DIR / '2_groupdro.csv', index=False)
pd.DataFrame([result_2])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 276.34it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,2_groupdro,bert_gdro_eta01_2ep,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [35]:
result_3 = run_model_checks('3_scrubbing', 'bert_scrubbing')
pd.DataFrame([result_3]).to_csv(OUTPUT_DIR / '3_scrubbing.csv', index=False)
pd.DataFrame([result_3])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 350.64it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,3_scrubbing,bert_scrubbing,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [36]:
result_4 = run_model_checks('4_oversampling', 'bert_oversample_only')
pd.DataFrame([result_4]).to_csv(OUTPUT_DIR / '4_oversampling.csv', index=False)
pd.DataFrame([result_4])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 536.77it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,4_oversampling,bert_oversample_only,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [37]:
result_5 = run_model_checks('5_focal_loss', 'bert_focal_loss')
pd.DataFrame([result_5]).to_csv(OUTPUT_DIR / '5_focal_loss.csv', index=False)
pd.DataFrame([result_5])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 425.68it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,5_focal_loss,bert_focal_loss,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [38]:
result_6 = run_model_checks('6_adversarial', 'bert_adversarial')
pd.DataFrame([result_6]).to_csv(OUTPUT_DIR / '6_adversarial.csv', index=False)
pd.DataFrame([result_6])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 611.21it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_adversarial/bert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,6_adversarial,bert_adversarial,True,split_bert,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,,True,True,True,True,"classifier.weight, classifier.bias",


In [39]:
result_7 = run_model_checks('7_label_smoothing', 'bert_label_smoothing')
pd.DataFrame([result_7]).to_csv(OUTPUT_DIR / '7_label_smoothing.csv', index=False)
pd.DataFrame([result_7])


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 163.74it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,7_label_smoothing,bert_label_smoothing,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [40]:
result_8 = run_model_checks('8_attribution_reg', 'bert_attr_reg')
pd.DataFrame([result_8]).to_csv(OUTPUT_DIR / '8_attribution_reg.csv', index=False)
pd.DataFrame([result_8])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 222.49it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_attr_reg/bert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,8_attribution_reg,bert_attr_reg,True,split_bert,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,,True,True,True,True,"classifier.weight, classifier.bias",


In [41]:
result_9 = run_model_checks('9_combined_scrub_gdro', 'bert_debiased_combo')
pd.DataFrame([result_9]).to_csv(OUTPUT_DIR / '9_combined_scrub_gdro.csv', index=False)
pd.DataFrame([result_9])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 243.55it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,9_combined_scrub_gdro,bert_debiased_combo,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [42]:
result_10 = run_model_checks('10_combined_best', 'combined_scrubbing_groupdro')
pd.DataFrame([result_10]).to_csv(OUTPUT_DIR / '10_combined_best.csv', index=False)
pd.DataFrame([result_10])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 210.11it/s, Materializing param=classifier.weight]                                      


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,10_combined_best,combined_scrubbing_groupdro,True,nested_hf,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",,,True,True,True,False,,


In [43]:
csv_paths = [OUTPUT_DIR / f'{model_name}.csv' for model_name, _ in MODELS]
summary = pd.concat([pd.read_csv(path) for path in csv_paths if path.exists()], ignore_index=True)
summary.to_csv(OUTPUT_DIR / 'model_diagnostics_summary.csv', index=False)
summary


,model_name,folder_name,model_dir_exists,layout,resolved_path,config_exists,main_weight_file,main_classifier_keys,sidecar_file,sidecar_classifier_keys,label_encoder_found,tokenizer_loaded,load_ok,missing_classifier_on_load,missing_keys,error
0,1_baseline,bert_9classes_final,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
1,2_groupdro,bert_gdro_eta01_2ep,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
2,3_scrubbing,bert_scrubbing,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
3,4_oversampling,bert_oversample_only,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
4,5_focal_loss,bert_focal_loss,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
5,6_adversarial,bert_adversarial,True,split_bert,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,NaN,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,NaN,True,True,True,True,"classifier.weight, classifier.bias",NaN
6,7_label_smoothing,bert_label_smoothing,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
7,8_attribution_reg,bert_attr_reg,True,split_bert,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,NaN,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,NaN,True,True,True,True,"classifier.weight, classifier.bias",NaN
8,9_combined_scrub_gdro,bert_debiased_combo,True,hf_root,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN
9,10_combined_best,combined_scrubbing_groupdro,True,nested_hf,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,"classifier.bias, classifier.weight",NaN,NaN,True,True,True,False,NaN,NaN


In [44]:
# === QUICK EVAL OF FIXED PROBLEM MODELS (6 and 8) ===
import gc
import re
import numpy as np
import joblib
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

def resolve_existing_path(candidates, label):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    searched = '\n'.join(str(Path(c)) for c in candidates)
    raise FileNotFoundError(f'{label} not found. Checked:\n{searched}')

TEST_CSV = resolve_existing_path([
    BASE_DIR / 'data' / 'processed' / 'test.csv',
    BASE_DIR.parent / 'data' / 'processed' / 'test.csv',
    BASE_DIR / 'resume-screening' / 'data' / 'processed' / 'test.csv',
    BASE_DIR.parent / 'resume-screening' / 'data' / 'processed' / 'test.csv',
], 'test.csv')

LABEL_MAP_CSV = resolve_existing_path([
    BASE_DIR / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    BASE_DIR / 'label_to_supercategory_v1.csv',
    BASE_DIR.parent / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    BASE_DIR / 'resume-screening' / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    BASE_DIR.parent / 'resume-screening' / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
], 'label_to_supercategory_v1.csv')

QUICK_EVAL_OUTPUT_DIR = OUTPUT_DIR / 'quick_eval_problem_models'
QUICK_EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROBLEM_MODELS = [
    ('6_adversarial', 'bert_adversarial', False),
    ('8_attribution_reg', 'bert_attr_reg', False),
]

SWAP_CITIES = ['Москва', 'Екатеринбург', 'Новосибирск', 'Краснодар', 'Воронеж']
BATCH_SIZE = 8
MAX_LENGTH = 128

CITY_PATTERNS = [
    'санкт-петербург', 'нижний новгород', 'ростов-на-дону',
    'набережные челны', 'магнитогорск', 'новосибирск',
    'екатеринбург', 'красноярск', 'волгоград', 'калининград',
    'владивосток', 'хабаровск', 'ставрополь', 'саратов',
    'челябинск', 'самара', 'казань', 'москва', 'омск',
    'воронеж', 'пермь', 'тюмень', 'томск', 'уфа',
    'тольятти', 'барнаул', 'иркутск', 'пенза', 'липецк',
    'кемерово', 'сочи', 'тверь', 'минск', 'алматы',
    'симферополь', 'ярославль', 'ульяновск', 'ижевск',
    'оренбург', 'мск', 'спб', 'питер',
]
escaped = [re.escape(c) for c in CITY_PATTERNS]
CITY_RE = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE)

print(f'Test CSV: {TEST_CSV.resolve()}')
print(f'Label map: {LABEL_MAP_CSV.resolve()}')
print(f'Quick eval output: {QUICK_EVAL_OUTPUT_DIR.resolve()}')


Test CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/data/processed/test.csv
Label map: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/data/processed/label_to_supercategory_v1.csv
Quick eval output: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/model_diagnostics/quick_eval_problem_models


In [45]:
def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ''
    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig and orig[0].isupper() else target_city.lower()
    return CITY_RE.sub(replacer, str(text))


def predict_batch(texts, model, tokenizer, batch_size=BATCH_SIZE):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
            preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, outputs
    return np.array(all_preds)


def find_eval_model_files(model_dir):
    model_dir = Path(model_dir)
    if (model_dir / 'config.json').exists():
        return 'hf', model_dir
    if (model_dir / 'bert' / 'config.json').exists() and any((model_dir / name).exists() for name in ['classifier.pt', 'model.pt']):
        return 'split_classifier', model_dir
    candidates = [sub for sub in model_dir.iterdir() if sub.is_dir() and (sub / 'config.json').exists()]
    if candidates:
        preferred = [sub for sub in candidates if sub.name == 'final']
        return 'hf', preferred[0] if preferred else sorted(candidates)[-1]
    return None, None


def _extract_state_dict(obj):
    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'classifier_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return None


def _load_split_classifier_weights(model, model_dir):
    model_dir = Path(model_dir)
    target_state = model.classifier.state_dict()
    for candidate in [model_dir / 'classifier.pt', model_dir / 'model.pt']:
        if not candidate.exists():
            continue
        obj = torch.load(candidate, map_location='cpu')
        state = _extract_state_dict(obj)
        if not isinstance(state, dict):
            continue
        remapped = {}
        for key, value in state.items():
            norm_key = key.replace('module.', '')
            if norm_key.startswith('classifier.'):
                norm_key = norm_key[len('classifier.'):]
            if norm_key in target_state and getattr(value, 'shape', None) == target_state[norm_key].shape:
                remapped[norm_key] = value
        if set(remapped.keys()) == set(target_state.keys()):
            model.classifier.load_state_dict(remapped, strict=True)
            return True
    return False


def load_model_for_quick_eval(model_name, model_dir):
    fmt, path = find_eval_model_files(model_dir)
    if fmt is None:
        print(f'  No model files found in {model_dir}')
        return None
    try:
        if fmt == 'split_classifier':
            bert_path = path / 'bert'
            tokenizer_path = path if (path / 'tokenizer.json').exists() else bert_path
            tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(bert_path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys) and not _load_split_classifier_weights(model, path):
                print(f'  Could not restore classifier head for {model_name}')
                return None
        else:
            tokenizer = AutoTokenizer.from_pretrained(str(path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys):
                print(f'  Missing classifier head for {model_name}: {sorted(missing_keys)}')
                return None

        le = None
        for le_path in [path / 'label_encoder.joblib', path.parent / 'label_encoder.joblib', model_dir / 'label_encoder.joblib']:
            if le_path.exists():
                le = joblib.load(le_path)
                break
        if le is None:
            print(f'  No label encoder for {model_name}')
            return None

        model = model.to(device).eval()
        return {'model': model, 'tokenizer': tokenizer, 'label_encoder': le}
    except Exception as exc:
        print(f'  Failed to load {model_name}: {type(exc).__name__}: {exc}')
        return None


In [46]:
df_eval = pd.read_csv(TEST_CSV)
mapping = pd.read_csv(LABEL_MAP_CSV)
l2s = dict(zip(mapping['label'], mapping['supercategory']))

quick_eval_rows = []

for model_name, folder_name, uses_scrubbing in PROBLEM_MODELS:
    print('\n' + '=' * 60)
    print(f'Quick eval: {model_name} ({folder_name})')
    model_dir = MODELS_BASE / folder_name
    loaded = load_model_for_quick_eval(model_name, model_dir)
    if loaded is None:
        quick_eval_rows.append({
            'model_name': model_name,
            'folder_name': folder_name,
            'load_ok': False,
            'accuracy': None,
            'macro_f1': None,
            'overall_flip_rate': None,
        })
        continue

    model = loaded['model']
    tokenizer = loaded['tokenizer']
    le = loaded['label_encoder']

    base_texts = df_eval['resume_text'].fillna('').astype(str).tolist()
    orig_preds = predict_batch(base_texts, model, tokenizer)

    any_flip = np.zeros(len(df_eval), dtype=bool)
    per_city = {}

    for swap_city in SWAP_CITIES:
        swapped = df_eval['resume_text'].apply(lambda x: swap_cities_in_text(x, swap_city)).fillna('').astype(str).tolist()
        swap_preds = predict_batch(swapped, model, tokenizer)
        flipped = swap_preds != orig_preds
        flip_rate = float(flipped.mean())
        any_flip |= flipped
        per_city[swap_city] = flip_rate
        print(f'  {swap_city}: {flip_rate:.3f} ({int(flipped.sum())}/{len(df_eval)})')
        del swapped, swap_preds, flipped
        gc.collect()

    y_true = le.transform(df_eval['label'].map(l2s).fillna('generic_it_ops'))
    acc = float(accuracy_score(y_true, orig_preds))
    f1 = float(f1_score(y_true, orig_preds, average='macro'))
    overall_flip = float(any_flip.mean())

    row = {
        'model_name': model_name,
        'folder_name': folder_name,
        'load_ok': True,
        'accuracy': acc,
        'macro_f1': f1,
        'overall_flip_rate': overall_flip,
    }
    for city, rate in per_city.items():
        row[f'flip_{city}'] = rate
    quick_eval_rows.append(row)
    print(f'  Acc={acc:.3f}  F1={f1:.3f}  Flip={overall_flip:.3f}')

    del model, tokenizer, le, orig_preds, any_flip
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

quick_eval_df = pd.DataFrame(quick_eval_rows)
quick_eval_df.to_csv(QUICK_EVAL_OUTPUT_DIR / 'quick_eval_problem_models_summary.csv', index=False)
quick_eval_df



Quick eval: 6_adversarial (bert_adversarial)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 665.85it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_adversarial/bert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Could not restore classifier head for 6_adversarial

Quick eval: 8_attribution_reg (bert_attr_reg)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 877.75it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_attr_reg/bert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Could not restore classifier head for 8_attribution_reg


,model_name,folder_name,load_ok,accuracy,macro_f1,overall_flip_rate
0,6_adversarial,bert_adversarial,False,None,None,None
1,8_attribution_reg,bert_attr_reg,False,None,None,None


In [47]:
print(f"{'Model':<22} {'Load':>6} {'Acc':>7} {'F1':>7} {'Flip%':>8}")
print('-' * 56)
for _, row in quick_eval_df.iterrows():
    if not row['load_ok']:
        print(f"{row['model_name']:<22} {'FAIL':>6}")
        continue
    print(f"{row['model_name']:<22} {'OK':>6} {row['accuracy']:>7.3f} {row['macro_f1']:>7.3f} {row['overall_flip_rate']:>8.3f}")


Model                    Load     Acc      F1    Flip%
--------------------------------------------------------
6_adversarial            FAIL
8_attribution_reg        FAIL
